In [1]:
import numpy as np
from sklearn.metrics import roc_auc_score, precision_score, recall_score, accuracy_score
import torch
import torch.nn as nn
from torch.autograd import Variable
import torch.nn.functional as F

In [2]:
import numpy as np

X1=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X1.npy")
X1=X1.reshape(X1.shape[0], 1, X1.shape[2], X1.shape[1])

X2=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X2.npy")
X2=X2.reshape(X2.shape[0], 1, X2.shape[2], X2.shape[1])

X3=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X3.npy")
X3=X3.reshape(X3.shape[0], 1, X3.shape[2], X3.shape[1])

X4=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X4.npy")
X4=X4.reshape(X4.shape[0], 1, X4.shape[2], X4.shape[1])

X5=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X5.npy")
X5=X5.reshape(X5.shape[0], 1, X5.shape[2], X5.shape[1])

X6=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X6.npy")
X6=X6.reshape(X6.shape[0], 1, X6.shape[2], X6.shape[1])

X_train=np.append(X1, X2,0)
X_train=np.append(X_train, X3,0)
X_train=np.append(X_train, X4,0)
X_train=np.append(X_train, X5,0)
X_train=np.append(X_train, X6,0)

X_train.shape

X1=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X1L.npy")

X2=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X2L.npy")

X3=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X3L.npy")

X4=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X4L.npy")

X5=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X5L.npy")

X6=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X6L.npy")

y_train=np.append(X1, X2,0)
y_train=np.append(y_train, X3,0)
y_train=np.append(y_train, X4,0)
y_train=np.append(y_train, X5,0)
y_train=np.append(y_train, X6,0)

y_train.shape

X7=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X7.npy")
X7=X7.reshape(X7.shape[0], 1, X7.shape[2], X7.shape[1])

X8=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X8.npy")
X8=X8.reshape(X8.shape[0], 1, X8.shape[2], X8.shape[1])

X_val=np.append(X7, X8,0)

X_val.shape

X7=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X7L.npy")

X8=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X8L.npy")

y_val=np.append(X7, X8,0)

y_val.shape

X9=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X9.npy")
X9=X9.reshape(X9.shape[0], 1, X9.shape[2], X9.shape[1])
X_test=X9
X_test.shape

X9=np.load(r"C:\Users\Hzaab\Desktop\intern\Data\X9L.npy")
y_test=X9
X_train.shape


(3456, 1, 1001, 22)

In [3]:

class EEGNetTorch(nn.Module):
    def __init__(self, nb_classes=4, Chans=22, Samples=1001, dropoutRate=0.5, kernLength=125, F1=8, D=2, F2=16, dropoutType="Dropout"):
        super().__init__()

        if dropoutType == "Dropout":
            DropoutLayer = nn.Dropout
        elif dropoutType == "SpatialDropout2D":
            DropoutLayer = nn.Dropout2d
        else:
            raise ValueError("dropoutType must be 'Dropout' or 'SpatialDropout2D'")

        #input: (batch, 1, Chans, Samples)

        # Block 1: Conv2D(F1, (1, kernLength), padding='same')
        self.conv_temporal = nn.Conv2d(in_channels=1, out_channels=F1,
            kernel_size=(1, kernLength),
            padding="same", bias=False,
        )
        self.bn1 = nn.BatchNorm2d(F1)

        # DepthwiseConv2D((Chans, 1), depth_multiplier=D)
        self.conv_spatial = nn.Conv2d(in_channels=F1, out_channels=F1 * D,
            kernel_size=(Chans, 1),
            groups=F1, bias=False,
        )
        self.bn2 = nn.BatchNorm2d(F1 * D)
        self.pool1 = nn.AvgPool2d(kernel_size=(1, 4))
        self.drop1 = DropoutLayer(dropoutRate)

        # Block 2: SeparableConv2D(F2, (1, 16), padding='same')
        self.sep_depthwise = nn.Conv2d(in_channels=F1 * D, out_channels=F1 * D,
            kernel_size=(1, 16),
            padding="same",
            groups=F1 * D, bias=False,
        )
        self.sep_pointwise = nn.Conv2d(in_channels=F1 * D, out_channels=F2,
            kernel_size=(1, 1),
            bias=False,
        )

        self.bn3 = nn.BatchNorm2d(F2)
        self.pool2 = nn.AvgPool2d(kernel_size=(1, 8))
        self.drop2 = DropoutLayer(dropoutRate)

        with torch.no_grad():
            dummy = torch.zeros(1, 1, Chans, Samples)
            out = self._features(dummy)
            self.flatten_dim = out.flatten(start_dim=1).shape[1]

        self.classifier = nn.Linear(self.flatten_dim, nb_classes)

    def _features(self, x):
        x = self.conv_temporal(x)
        x = self.bn1(x)

        x = self.conv_spatial(x)

        x = self.bn2(x)
        x = F.elu(x)
        x = self.pool1(x)
        x = self.drop1(x)

        x = self.sep_depthwise(x)
        x = self.sep_pointwise(x)
        x = self.bn3(x)
        x = F.elu(x)
        x = self.pool2(x)
        x = self.drop2(x)

        return x

    def forward(self, x):
        x = self._features(x)
        x = torch.flatten(x, start_dim=1)
        x = self.classifier(x)
        return x

In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
net = EEGNetTorch(
    nb_classes=4,
    Chans=22,
    Samples=1001,
    dropoutRate=0.5,
    kernLength=125,
    F1=8,
    D=2,
    F2=16,
    dropoutType="Dropout"
).float().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)

x = torch.randn(2, 1, 22, 1001).float().to(device)
print(net(x).shape)

c:\Users\Hzaab\Desktop\intern\.venv\Lib\site-packages\torch\nn\modules\conv.py:560: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1091.)
  return F.conv2d(


torch.Size([2, 4])


In [5]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
import numpy as np
import torch

def evaluate(model, X, Y, params, batch_size=64):
    model.eval()

    all_probs = []
    all_preds = []

    with torch.no_grad():
        for s in range(0, len(X), batch_size):
            e = min(s + batch_size, len(X))

            inputs = torch.from_numpy(X[s:e]).float()

            # convert (batch, 1, 1001, 22) -> (batch, 1, 22, 1001)
            if inputs.shape[2] == 1001 and inputs.shape[3] == 22:
                inputs = inputs.permute(0, 1, 3, 2)

            inputs = inputs.cuda(0)

            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)

            all_probs.append(probs.cpu().numpy())
            all_preds.append(probs.argmax(dim=1).cpu().numpy())

    probs = np.concatenate(all_probs, axis=0)
    pred_labels = np.concatenate(all_preds, axis=0)

    print("Y shape:", Y.shape)
    print("pred shape:", pred_labels.shape)
    print("probs shape:", probs.shape)

    results = []

    for param in params:
        if param == "acc":
            results.append(accuracy_score(Y, pred_labels))

        elif param == "auc":
            try:
                results.append(
                    roc_auc_score(
                        Y,
                        probs,
                        multi_class="ovo",
                        labels=np.arange(probs.shape[1])
                    )
                )
            except ValueError:
                results.append(None)

        elif param == "fmeasure":
            results.append(f1_score(Y, pred_labels, average="macro"))

    model.train()
    return results

In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)

y_val_encoded = le.transform(y_val)
y_test_encoded = le.transform(y_test)




In [9]:
batch_size = 32

for epoch in range(20):  # loop over the dataset multiple times
    print("\nEpoch ", epoch)
    
    running_loss = 0.0
    for i in range(len(X_train)//batch_size):
        s = i*batch_size
        e = i*batch_size+batch_size
        
        inputs = torch.from_numpy(X_train[s:e]).float()

        if inputs.shape[2] == 1001 and inputs.shape[3] == 22:
                inputs = inputs.permute(0, 1, 3, 2)

        inputs = inputs.cuda(0)
        labels = torch.tensor(
            y_train_encoded[s:e],
            dtype=torch.long,
            device="cuda"
        )
        
        # wrap them in Variable
        inputs, labels = Variable(inputs.cuda(0)), Variable(labels.cuda(0))

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        
        
        optimizer.step()
        
        running_loss += loss.item()
    
    # Validation accuracy
    params = ["acc", "auc", "fmeasure"]
    print(params)
    print("Training Loss ", running_loss)
    print("Train - ", evaluate(net, X_train, y_train_encoded, params))
    print("Validation - ", evaluate(net, X_val, y_val_encoded, params))
print("Test - ", evaluate(net, X_test, y_test_encoded, params))


Epoch  0
['acc', 'auc', 'fmeasure']
Training Loss  151.42574071884155
Y shape: (3456,)
pred shape: (3456,)
probs shape: (3456, 4)
Train -  [0.3072916666666667, np.float64(0.5807268781971594), 0.28655137021138066]
Y shape: (1152,)
pred shape: (1152,)
probs shape: (1152, 4)
Validation -  [0.23524305555555555, np.float64(0.5276702755272634), 0.23225312356166267]

Epoch  1
['acc', 'auc', 'fmeasure']
Training Loss  148.22462952136993
Y shape: (3456,)
pred shape: (3456,)
probs shape: (3456, 4)
Train -  [0.3133680555555556, np.float64(0.5872026329232396), 0.28732991978282596]
Y shape: (1152,)
pred shape: (1152,)
probs shape: (1152, 4)
Validation -  [0.2508680555555556, np.float64(0.5357168692129629), 0.24512638922553787]

Epoch  2
['acc', 'auc', 'fmeasure']
Training Loss  146.67220854759216
Y shape: (3456,)
pred shape: (3456,)
probs shape: (3456, 4)
Train -  [0.3336226851851852, np.float64(0.5933365684477881), 0.29821446402342455]
Y shape: (1152,)
pred shape: (1152,)
probs shape: (1152, 4)
V